# AutoAssess — Train YOLOv8-seg on VehiDE (Colab)

Trains the damage-detection model on the **VehiDE** dataset (7 structural/cosmetic damage classes — scratch, dent, torn, broken_lamp, missing_part, punctured, shattered_glass) instead of CarDD's 6, to cover damage types CarDD has no equivalent for.

**Before running:** Runtime -> Change runtime type -> select a GPU (T4 is fine).

**You'll need a Kaggle API token.** Get one at kaggle.com -> Settings -> API -> Create New Token (downloads `kaggle.json`). Add it as a Colab secret (key icon in the left sidebar) named `KAGGLE_JSON` containing the full file contents, or upload `kaggle.json` when prompted in the cell below.

## 1. Clone the repo and install dependencies

In [ ]:
import os
os.chdir("/content")
!rm -rf /content/autoassess
!git clone https://github.com/DhyeyTandel/autoassess.git
%cd /content/autoassess
!pip install -q -e .
!pip install -q kaggle

## 2. Kaggle credentials

Tries a Colab secret named `KAGGLE_JSON` first; falls back to a manual file upload if the secret isn't set.

In [ ]:
import json
import os
from pathlib import Path

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
kaggle_json_path = kaggle_dir / "kaggle.json"

wrote_from_secret = False
try:
    from google.colab import userdata
    raw = userdata.get("KAGGLE_JSON")
    json.loads(raw)  # validate it's real JSON before writing
    kaggle_json_path.write_text(raw)
    wrote_from_secret = True
    print("Wrote ~/.kaggle/kaggle.json from Colab secret KAGGLE_JSON.")
except Exception as e:
    print(f"Could not read Colab secret KAGGLE_JSON ({e!r}). Falling back to file upload.")

if not wrote_from_secret:
    from google.colab import files
    print("Upload your kaggle.json file:")
    uploaded = files.upload()
    fname = next(iter(uploaded))
    kaggle_json_path.write_bytes(uploaded[fname])

os.chmod(kaggle_json_path, 0o600)
print("Kaggle credentials ready at", kaggle_json_path)

Could not read Colab secret KAGGLE_JSON (SecretNotFoundError('Secret KAGGLE_JSON does not exist.')). Falling back to file upload.
Upload your kaggle.json file:


## 3. Download + convert VehiDE

Same scripts used locally — `scripts/download_vehide.py` (Kaggle API) and `autoassess-convert-vehide` (VIA JSON -> YOLO-seg format).

In [ ]:
!python scripts/download_vehide.py --config configs/base.yaml

In [ ]:
!autoassess-convert-vehide --config configs/base.yaml --dataset-config configs/vehide.yaml

## 4. Train YOLOv8-seg on VehiDE

Uses the repo's own defaults (`yolov8s-seg`, 50 epochs, batch 8, imgsz 640) — same training script (`train_yolo.py`) as the CarDD/parts models, just pointed at `configs/vehide.yaml`. `device 0` is Colab's GPU.

**Colab's local disk is ephemeral** — if the runtime disconnects or recycles mid-training (common on free tier), everything written to `/content/autoassess/runs/` is lost with it. The cells below mount Drive, verify the mount is real (not just requested), and mirror checkpoints to Drive continuously during training so a dead runtime doesn't lose the run.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Hard fail here rather than discovering after a multi-hour run that nothing
# was actually persisted. A previous run silently trained to local disk only
# because this check didn't exist — the runtime recycled and the weights were gone.
assert os.path.ismount('/content/drive'), (
    'Drive did not mount — re-run this cell and approve the auth prompt before continuing.'
)

DRIVE_BACKUP_DIR = '/content/drive/MyDrive/autoassess_runs/vehide_seg_v1'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
print(f'Drive mounted and writable. Backing up to: {DRIVE_BACKUP_DIR}')

In [ ]:
import subprocess, sys

# Mirrors runs/vehide_seg_v1/ -> Drive every 60s in the background for the rest of this
# session, so mid-training checkpoints (weights/last.pt) survive a runtime crash even if
# the final packaging cell never runs. rsync only copies changed bytes, so this is cheap.
mirror_script = '''
import subprocess, time, pathlib
src = pathlib.Path("/content/autoassess/runs/vehide_seg_v1")
dst = "/content/drive/MyDrive/autoassess_runs/vehide_seg_v1/"
while True:
    if src.exists():
        subprocess.run(["rsync", "-a", str(src) + "/", dst], capture_output=True)
    time.sleep(60)
'''
with open('/content/_mirror_to_drive.py', 'w') as f:
    f.write(mirror_script)

mirror_proc = subprocess.Popen([sys.executable, '/content/_mirror_to_drive.py'])
print(f'Background Drive mirror started (PID {mirror_proc.pid}). '
      'Checkpoints will appear in Drive within ~60s of being written locally.')

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
!autoassess-train-yolo \
  --dataset-config configs/vehide.yaml \
  --model yolov8s-seg.pt \
  --epochs 50 \
  --batch 16 \
  --imgsz 640 \
  --device 0 \
  --name vehide_seg_v1

### If the runtime disconnects mid-training

**Same runtime still alive** (you just interrupted the cell, or it errored but the VM is up): just re-run the training cell above with `--resume` added.

**Runtime fully disconnected/recycled** (local disk wiped): re-run cells 1-2 (clone) and cells 6-7 (data), then re-run the two Drive cells above (9-10), then run the cell below — it restores your last checkpoint from Drive and resumes training automatically.

In [ ]:
import os, subprocess

DRIVE_RUN_DIR = "/content/drive/MyDrive/autoassess_runs/vehide_seg_v1"
LOCAL_RUN_DIR = "/content/autoassess/runs/vehide_seg_v1"

assert os.path.ismount("/content/drive"), "Re-run the Drive mount cell first."
assert os.path.isdir(DRIVE_RUN_DIR), (
    f"No backup found at {DRIVE_RUN_DIR} — nothing to resume from. "
    "Run the normal training cell instead."
)

os.makedirs(LOCAL_RUN_DIR, exist_ok=True)
subprocess.run(["rsync", "-a", DRIVE_RUN_DIR + "/", LOCAL_RUN_DIR + "/"], check=True)
print(f"Restored checkpoint from Drive into {LOCAL_RUN_DIR}")

last_ckpt = os.path.join(LOCAL_RUN_DIR, "weights", "last.pt")
assert os.path.exists(last_ckpt), f"Restored dir has no weights/last.pt — check {DRIVE_RUN_DIR}"
print(f"Found checkpoint: {last_ckpt}")

!autoassess-train-yolo --dataset-config configs/vehide.yaml --model yolov8s-seg.pt \
  --epochs 50 --batch 16 --imgsz 640 --device 0 --name vehide_seg_v1 --resume

## 5. Check results

In [ ]:
import json

metrics = json.loads(Path("runs/vehide_seg_v1/metrics.json").read_text())
print(json.dumps({k: v for k, v in metrics.items() if k != "epoch_wall_times_seconds"}, indent=2))

## 6. Package weights + metrics for download

The background mirror (started in section 4) already copies checkpoints to `/content/drive/MyDrive/autoassess_runs/vehide_seg_v1/` throughout training — so even if this cell is never run, the weights are already safe in Drive.

This cell does one final sync, then zips **from Drive** (falling back to local disk only if Drive is somehow unavailable) and offers it as a direct browser download too, so you have both a Drive copy and a local zip. Extract the zip into `runs/vehide_seg_v1/` in your local repo so eval scripts and `app.py` can pick it up.

In [ ]:
import os, shutil, subprocess

LOCAL_RUN_DIR = "/content/autoassess/runs/vehide_seg_v1"
DRIVE_RUN_DIR = "/content/drive/MyDrive/autoassess_runs/vehide_seg_v1"

# Final sync in case training just finished and the last 60s mirror tick hasn't fired yet.
if os.path.ismount("/content/drive") and os.path.isdir(LOCAL_RUN_DIR):
    os.makedirs(DRIVE_RUN_DIR, exist_ok=True)
    subprocess.run(["rsync", "-a", LOCAL_RUN_DIR + "/", DRIVE_RUN_DIR + "/"], check=True)
    print(f"Final sync to Drive complete: {DRIVE_RUN_DIR}")

# Prefer Drive as the source of truth (survives this runtime dying); fall back to local
# disk only if Drive genuinely never mounted this session.
source_dir = DRIVE_RUN_DIR if os.path.isdir(DRIVE_RUN_DIR) else LOCAL_RUN_DIR
assert os.path.isdir(source_dir), (
    f"No run directory found at either {DRIVE_RUN_DIR} or {LOCAL_RUN_DIR} — "
    "training may not have produced output yet."
)
print(f"Packaging from: {source_dir}")

shutil.make_archive("/content/vehide_seg_v1_results", "zip", source_dir)
print("Packaged: /content/vehide_seg_v1_results.zip")

from google.colab import files
files.download("/content/vehide_seg_v1_results.zip")